# Paper 1 (brain paper): figures

Run `paper1_analysis.ipynb` first. Fig. 3, weighted structure against degree and across the lifespan; Fig. 5, memory and its structural predictors (layout of v1); Fig. 6, interhemispheric weight and memory across the lifespan. Development and aging are split at 32 years, the minimum of the running mean of memory. PNAS single column (8.7 cm), 400 dpi, PDF and PNG, saved to `paper/figures/p1/`. Memory capacity (MC) is always the simulated reservoir.

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

REPO = Path.cwd().resolve().parent                       # this notebook lives in paper/
DATA, D3, D1, OUT = REPO / "paper/data", REPO / "paper/data/v3", REPO / "paper/data/p1", REPO / "paper/figures/p1"
OUT.mkdir(parents=True, exist_ok=True)
W1 = 3.42                                                # PNAS single column, 8.7 cm; figures are saved at exactly this width
INK, INK2 = "#1a1a1a", "#666666"
ORDER = ["dHCP", "BCP", "CALM", "RED", "ACE", "HCPd", "HCPya", "camCAN", "HCPa"]
COH = dict(zip(ORDER, ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b", "#7f7f7f", "#bcbd22", "#17becf"]))  # cohort colours of v1

sns.set_theme(context="paper", style="ticks")
mpl.rcParams.update({
    "font.family": "sans-serif", "font.sans-serif": ["Arial", "Helvetica", "Liberation Sans", "DejaVu Sans"],
    "mathtext.fontset": "stix", "font.size": 7.5, "axes.labelsize": 7.5, "axes.titlesize": 7.5,
    "xtick.labelsize": 7, "ytick.labelsize": 7, "legend.fontsize": 6.8, "legend.title_fontsize": 6.8,
    "axes.linewidth": 0.6, "axes.edgecolor": INK, "axes.labelcolor": INK, "xtick.color": INK, "ytick.color": INK,
    "text.color": INK, "xtick.major.width": 0.6, "ytick.major.width": 0.6, "xtick.minor.width": 0.45,
    "ytick.minor.width": 0.45, "xtick.major.size": 2.5, "ytick.major.size": 2.5, "xtick.minor.size": 1.4,
    "ytick.minor.size": 1.4, "xtick.major.pad": 1.6, "ytick.major.pad": 1.6, "axes.labelpad": 1.8,
    "lines.linewidth": 1.0, "legend.frameon": False, "legend.handlelength": 1.3, "legend.handletextpad": 0.4,
    "legend.borderaxespad": 0.2, "legend.labelspacing": 0.25, "pdf.fonttype": 42, "savefig.dpi": 400})


def letter(ax, s, x=-0.3, y=1.0):
    ax.text(x, y, s, transform=ax.transAxes, fontsize=9.5, fontweight="bold", va="bottom", ha="left")


def save(fig, name):
    """Exact figure size (no tight bounding box), so the text keeps its size in the column."""
    for ext in ("pdf", "png"):
        fig.savefig(OUT / f"{name}.{ext}", dpi=400)


def r_(a, b):
    return float(np.corrcoef(a, b)[0, 1])


def age_means(d, col):
    """Mean of col per cohort and age: 1-year bins, 3-month bins below 2 years (as in v1)."""
    a = np.where(d.age < 2, np.floor(d.age * 4) / 4 + 0.125, np.floor(d.age) + 0.5)
    return d.assign(abin=a).groupby(["cohort", "abin"])[col].mean().reset_index()


def trajectories(ax, d, col, ylabel):
    m = age_means(d, col)
    for c in ORDER:
        g = m[m.cohort == c]
        ax.plot(g.abin, g[col], "-o", color=COH[c], ms=1.7, lw=0.55, mew=0)
    ax.set(xlabel="age (years)", ylabel=ylabel, xlim=(-2, 92), xticks=[0, 20, 40, 60, 80])

DEV, AGI = "#E6A117", "#6A51A3"                         # development and aging, as in v1
DISP, TRI, HOM, RATIO, NULLG = "#08519c", "#e6550d", "#1b9e77", "#d62728", "#636363"   # disparity / order 2, triangles / odd walks, interhemispheric, disparity ratio, null models

In [ ]:
d = pd.read_csv(D3 / "subjects_v3.csv").set_index("sid")                 # age, cohort, structure, simulated MC
d = d.join(pd.read_csv(D1 / "race_metrics.csv").set_index("sid")[["cii"]])
d = d.join(pd.read_csv(D1 / "lifespan_equalised.csv").set_index("sid")[["MC_eq", "MC_hom_eq", "frac", "top_inter", "top_intra"]])
N1 = json.load(open(D1 / "numbers_p1.json")); N3 = json.load(open(D3 / "numbers_v3.json"))
orders = pd.read_csv(D1 / "cii_orders_by_degree.csv"); nodes = pd.read_csv(D1 / "cii_nodes_sample.csv")
hk = pd.read_csv(D1 / "node_homotopic_share.csv"); nl = pd.read_csv(D1 / "nulls_sim.csv")
uk = pd.read_csv(D3 / "upsilon_k.csv"); nodes_k = pd.read_csv(D3 / "nodes_sample.csv")


def logaxis(axis, ticks=(1, 3, 10, 30, 90), fmt=None):
    axis.set_major_locator(mpl.ticker.FixedLocator(ticks)); axis.set_minor_locator(mpl.ticker.NullLocator())
    axis.set_major_formatter(mpl.ticker.FixedFormatter(fmt) if fmt else mpl.ticker.ScalarFormatter())


def running(d, col, w=5):
    """Mean of col per year of age, smoothed with a centred running mean over w years."""
    m = d.groupby("age")[col].mean(); return m.rolling(w, center=True, min_periods=3).mean()

## Fig. 3. Weighted structure against degree (top) and across the lifespan (bottom)
Disparity, subgraph centrality and homotopic share: (a-c) against degree, with null models where they exist; (d-f) per-age means of each cohort.

In [ ]:
fig = plt.figure(figsize=(W1, 3.05))
gs = fig.add_gridspec(2, 3, left=0.125, right=0.985, top=0.955, bottom=0.12, wspace=0.78, hspace=0.62)

# (a) disparity against degree, with the null model; Upsilon = kY lies between 1 and k
ax = fig.add_subplot(gs[0, 0])
ax.scatter(nodes_k.k, nodes_k.Ups, s=0.35, color="#6baed6", alpha=0.15, lw=0, rasterized=True)
k = np.arange(1, 90.0); mu = 2 * k / (k + 1)
sd = k * np.sqrt(np.clip((20 + 4 * k) / ((k + 1) * (k + 2) * (k + 3)) - 4 / (k + 1) ** 2, 0, None))
ax.fill_between(k, np.clip(mu - 2 * sd, 1, None), np.minimum(mu + 2 * sd, k), color="#bdbdbd", alpha=0.85, lw=0, label="null $\\pm2\\sigma$")
ax.plot(k, mu, color=NULLG, lw=0.7); ax.plot(k, k, color=RATIO, lw=0.8, ls="--", label="$\\Upsilon=k$")
g = uk[uk["size"] >= 30]; ax.plot(g.k, g["mean"], color=DISP, lw=1.3, label=f"$\\propto k^{{{N3['upsilon_alpha']:.2f}}}$")
ax.set(xscale="log", yscale="log", xlabel="degree $k$", ylabel="$\\Upsilon=kY$", xlim=(1, 90), ylim=(1, 90))
logaxis(ax.xaxis); logaxis(ax.yaxis); ax.legend(loc="upper left", fontsize=5.6, handlelength=1.1, borderaxespad=0.1); letter(ax, "a", x=-0.5)

# (b) share of subgraph centrality captured by the walk expansion truncated at orders 2 to 5 (mean; interquartile range over nodes)
ax = fig.add_subplot(gs[0, 1]); km = np.sqrt(orders.k_lo * (orders.k_hi - 1).clip(lower=orders.k_lo)).values
for m in (2, 3):
    nodes[f"s{m}"] = 100 * (nodes[f"ord{m}"] - 1) / (nodes.C - 1)
nodes["bin"] = pd.cut(nodes.k, list(orders.k_lo) + [90], right=False); q = nodes.groupby("bin", observed=True)
kq = km[[list(orders.k_lo).index(int(iv.left)) for iv in q.size().index]]
for m, c, ls, lab in ((2, DISP, "-", "order 2"), (3, TRI, "-", "orders 2-3"), (4, "#7f7f7f", "--", "orders 2-4"), (5, "#b0b0b0", ":", "orders 2-5")):
    ax.plot(km, 100 * orders[f"frac{m}"], ls, color=c, lw=0.9, marker="o" if m <= 3 else None, ms=1.6, label=lab)
    if m <= 3:
        ax.fill_between(kq, q[f"s{m}"].quantile(0.25).values, q[f"s{m}"].quantile(0.75).values, color=c, alpha=0.18, lw=0)
ax.set(xscale="log", xlim=(1.5, 60), ylim=(84, 100.6), xlabel="degree $k$", ylabel="share of $C_{ii}-1$ (%)")
logaxis(ax.xaxis, (2, 5, 10, 20, 50)); ax.legend(loc="lower left", fontsize=5.6, handlelength=1.4, borderaxespad=0.1); letter(ax, "b", x=-0.62)

# (c) share of a region's squared weight carried by its homotopic connection, against degree (mean and 95% CI), and the exchangeable null 1/k
ax = fig.add_subplot(gs[0, 2]); g = hk[hk["count"] >= 50]; ci = 1.96 * g["std"] / np.sqrt(g["count"])
ax.fill_between(g.k, g["mean"] - ci, g["mean"] + ci, color=HOM, alpha=0.3, lw=0)
ax.plot(g.k, g["mean"], "-o", color=HOM, lw=0.9, ms=1.6, label="homotopic")
ax.plot(g.k, 1 / g.k, color=NULLG, lw=0.8, ls="--", label="null $1/k$")
ax.set(xscale="log", yscale="log", xlim=(2, 60), ylim=(0.015, 0.8), xlabel="degree $k$", ylabel="homotopic share $h_i$")
logaxis(ax.xaxis, (2, 5, 10, 20, 50)); logaxis(ax.yaxis, (0.02, 0.1, 0.5), ["0.02", "0.1", "0.5"])
ax.legend(loc="lower left", fontsize=5.6, handlelength=1.4, borderaxespad=0.1); letter(ax, "c", x=-0.62)

# (d-f) lifespan trajectories: per-age means of each cohort
for j, (col, lab) in enumerate((("Ratio", "$\\langle\\Upsilon/\\Upsilon_{null}\\rangle$"), ("cii", "$\\langle C_{ii}\\rangle$"), ("hom_F2", "homotopic share $h$"))):
    ax = fig.add_subplot(gs[1, j]); trajectories(ax, d, col, lab); ax.set_xticks([0, 40, 80])
    if col == "cii":
        ax.yaxis.set_major_formatter(mpl.ticker.FormatStrFormatter("%.2f"))
    letter(ax, "def"[j], x=-0.62 if j else -0.5)
for a in fig.axes:
    sns.despine(ax=a)
save(fig, "fig3_structure")

## Fig. 5. Reservoir memory across the lifespan and its structural predictors
(a) MC and disparity ratio against age, running mean and a linear fit per regime; (b) MC against the stable rank; (c) correlation with MC in development and aging; (d) partial correlation given the stable rank; (e) null models.

In [ ]:
fig = plt.figure(figsize=(W1, 4.1))
top = fig.add_gridspec(1, 2, left=0.13, right=0.87, top=0.9, bottom=0.6, wspace=0.75)
bot = fig.add_gridspec(1, 2, left=0.335, right=0.70, top=0.47, bottom=0.17, wspace=0.22, width_ratios=[1.0, 0.72])
bote = fig.add_gridspec(1, 1, left=0.80, right=0.985, top=0.47, bottom=0.17)
dev, agi = d[d.age <= 32], d[d.age > 32]; SPLIT = 32

# (a) memory and disparity ratio across the lifespan: running mean of the per-year means, and a linear fit in each regime
ax = fig.add_subplot(top[0])
ax.scatter(d.age, d.MC, s=0.5, color="#bdbdbd", alpha=0.3, lw=0, rasterized=True)
for g, c, lab, lo, hi in ((dev, DEV, "development", 0, SPLIT), (agi, AGI, "aging", SPLIT, 90)):
    b = np.polyfit(g.age, g.MC, 1); x = np.array([lo, hi]); ax.plot(x, np.polyval(b, x), color=c, lw=1.6, label=lab)
ax.axvline(SPLIT, color=INK2, lw=0.5, ls=":")
ax.set(xlabel="age (years)", ylabel="MC", xlim=(-2, 92), xticks=[0, 40, 80], ylim=(6.5, 12.6))
fig.legend(*ax.get_legend_handles_labels(), loc="upper center", bbox_to_anchor=(0.5, 1.0), ncol=2, columnspacing=1.2, handlelength=1.2)
ax2 = ax.twinx()
for g, lo, hi in ((dev, 0, SPLIT), (agi, SPLIT, 90)):
    b = np.polyfit(g.age, g.Ratio, 1); x = np.array([lo, hi]); ax2.plot(x, np.polyval(b, x), color=RATIO, lw=1.2, ls="--")
ax2.set_ylim(1.3, 1.9); ax2.set_ylabel("$\\langle\\Upsilon/\\Upsilon_{null}\\rangle$", color=RATIO); ax2.tick_params(axis="y", colors=RATIO)
ax2.spines["right"].set_color(RATIO); ax2.spines["top"].set_visible(False); letter(ax, "a", x=-0.45)

# (b) memory against the stable rank, by regime
ax = fig.add_subplot(top[1])
for g, c in ((dev, DEV), (agi, AGI)):
    ax.scatter(g.rs, g.MC, s=0.7, color=c, alpha=0.35, lw=0, rasterized=True)
    b = np.polyfit(np.log(g.rs), g.MC, 1); x = np.linspace(g.rs.min(), g.rs.max(), 50); ax.plot(x, np.polyval(b, np.log(x)), color=c, lw=1.1)
rr_ = N1["race_regime"]["rs"]
ax.text(0.04, 0.95, f"$r$ = {rr_['development']:.2f}", color=DEV, transform=ax.transAxes, va="top", fontweight="bold")
ax.text(0.04, 0.83, f"$r$ = {rr_['aging']:.2f}", color=AGI, transform=ax.transAxes, va="top", fontweight="bold")
ax.set(xscale="log", xlabel="stable rank $r_s$", ylabel="MC", ylim=(6.5, 12.6)); logaxis(ax.xaxis, (3, 4, 6, 9)); letter(ax, "b", x=-0.42)

# (c) correlation with memory in development and aging; (d) partial correlation given the stable rank
ROWS = [("rs", "stable rank (ord. 2)"), ("tr3", "triangles (ord. 3)"), ("cii", "subgraph centrality"), ("comm", "communicability"),
        ("Ratio", "disparity ratio"), ("s_raw", "strength"), ("k", "degree"), ("avgctrl", "avg. controllability")]
y = np.arange(len(ROWS))[::-1].astype(float); y[4:] -= 0.5; y[7:] -= 0.5           # walks, ingredients, Gramian
RR = N1["race_regime"]; PR = N1["race"]
ax = fig.add_subplot(bot[0])
for yi, (key, lab) in zip(y, ROWS):
    a_, b_ = RR[key]["development"], RR[key]["aging"]
    ax.plot([a_, b_], [yi, yi], color="#c8c8c8", lw=1.6, solid_capstyle="round", zorder=1)
    ax.plot(a_, yi, "o", color=DEV, ms=3.4, zorder=2); ax.plot(b_, yi, "o", color=AGI, ms=3.4, zorder=2)
ax.axvline(0, color=INK2, lw=0.5); ax.set_yticks(y); ax.set_yticklabels([l for _, l in ROWS], fontsize=6.4); ax.tick_params(axis="y", length=0)
ax.set(xlim=(-0.45, 1.02), xticks=[0, 0.5, 1], xlabel="$r$ with MC", ylim=(y.min() - 0.6, y.max() + 0.6)); ax.set_xticklabels(["0", "0.5", "1"])
letter(ax, "c", x=-1.3)
ax = fig.add_subplot(bot[1], sharey=fig.axes[-1])
for yi, (key, lab) in zip(y, ROWS):
    if key == "rs":
        ax.text(0, yi, "-", ha="center", va="center", color=INK2); continue
    p = PR[key]["partial"]; ax.plot([0, p], [yi, yi], color=INK2, lw=1.0); ax.plot(p, yi, "o", color=INK, ms=3.0)
ax.axvline(0, color=INK2, lw=0.5); ax.tick_params(axis="y", left=False, labelleft=False)
ax.set(xlim=(-0.6, 0.45), xticks=[-0.4, 0, 0.4], xlabel="partial $r$ | $r_s$"); ax.set_xticklabels(["-0.4", "0", "0.4"]); letter(ax, "d", x=-0.12)

# (e) null models: change in memory, observed (boxes) and predicted from the change in stable rank (diamonds)
ax = fig.add_subplot(bote[0]); MOD = ["Uniform", "BrokenStick", "Reshuffle", "InterHalf"]
LAB = ["Uniform", "Broken stick", "Reshuffle", "Inter. $\\times\\frac{1}{2}$"]; PAL = [mpl.colors.to_rgba(DISP, 0.75), mpl.colors.to_rgba(DISP, 0.4), mpl.colors.to_rgba(NULLG, 0.35), mpl.colors.to_rgba(HOM, 0.7)]
bp = ax.boxplot([nl[nl.model == m].dMC.values for m in MOD], positions=range(4), widths=0.55, patch_artist=True, showfliers=False,
                medianprops=dict(color=INK, lw=0.9), whiskerprops=dict(color=INK2, lw=0.6), capprops=dict(color=INK2, lw=0.6), boxprops=dict(lw=0.6, edgecolor=INK2))
for patch, c in zip(bp["boxes"], PAL):
    patch.set_facecolor(c)
ax.plot(range(4), [nl[nl.model == m].dMC_pred.median() for m in MOD], "D", color="white", mec=INK, mew=0.7, ms=3.2, label="predicted")
ax.axhline(0, color=INK2, lw=0.5); ax.set_xticks(range(4)); ax.set_xticklabels(LAB, rotation=62, ha="right", fontsize=6.0, rotation_mode="anchor")
ax.set(ylabel="$\\Delta$MC (null $-$ real)", xlim=(-0.6, 3.6)); ax.tick_params(axis="x", length=0)
ax.legend(loc="lower right", fontsize=6.0, handletextpad=0.2, borderaxespad=0.1); letter(ax, "e", x=-0.6)
for a in fig.axes:
    if a is not ax2:
        sns.despine(ax=a)
save(fig, "fig5_memory")

## Fig. 6. Interhemispheric weight, walk lengths and the strongest connections
(a) MC against the homotopic share, coloured by age; (b) closed walks of each length: correlation with MC and partial correlation given the stable rank; (c) share of the squared weight on the strongest connection between and within the hemispheres, coloured by age.

In [ ]:
fig = plt.figure(figsize=(W1, 1.85))
gs = fig.add_gridspec(1, 3, left=0.105, right=0.955, top=0.8, bottom=0.27, wspace=0.75, width_ratios=[1, 1, 1.05])
AGE = dict(c=d.age, cmap="viridis", vmin=0, vmax=90, s=0.6, lw=0, alpha=0.75, rasterized=True)

# (a) memory against the homotopic share, all connectomes, coloured by age
ax = fig.add_subplot(gs[0]); sc = ax.scatter(d.hom_F2, d.MC, **AGE)
ax.text(0.05, 0.05, f"$r$ = {N1['interhemispheric']['r_MC_h']['all']:.2f}", transform=ax.transAxes, ha="left", va="bottom")
ax.set(xlabel="homotopic share $h$", ylabel="MC", xticks=[0, 0.4, 0.8]); letter(ax, "a", x=-0.55)
cax = fig.add_axes([0.105, 0.9, 0.2, 0.03]); cb = fig.colorbar(sc, cax=cax, orientation="horizontal")
cb.set_ticks([0, 45, 90]); cb.ax.tick_params(labelsize=5.8, length=1.5, pad=1, width=0.4); cb.outline.set_linewidth(0.3)
cax.text(1.06, 0.5, "age (years)", transform=cax.transAxes, va="center", fontsize=6.2)

# (b) closed walks of each length: correlation with memory, and partial correlation given the stable rank
ax = fig.add_subplot(gs[1]); R = N1["race"]; n = np.arange(2, 9)
rv = np.array([R[f"tr{m}"]["r"] for m in n]); pv = np.array([R[f"tr{m}"]["partial"] for m in n])
ax.plot(n, rv, color="#b0b0b0", lw=0.7, zorder=1); ax.plot(n[1:], pv[1:], color="#b0b0b0", lw=0.7, ls="--", zorder=1)
for m, a_, b_ in zip(n, rv, pv):
    c = TRI if m % 2 else DISP
    ax.plot(m, a_, "o", color=c, ms=3.2, zorder=2)
    if m > 2:
        ax.plot(m, b_, "o", mfc="white", mec=c, mew=0.9, ms=3.2, zorder=3)
ax.axhline(0, color=INK2, lw=0.5)
ax.plot([], [], "o", color=INK, ms=3.2, label="$r$"); ax.plot([], [], "o", mfc="white", mec=INK, ms=3.2, label="partial | $r_s$")
ax.set(xlabel="walk length $n$", ylabel="$r$ with MC", xticks=list(n), ylim=(-0.62, 1.05))
ax.legend(loc="lower center", bbox_to_anchor=(0.5, 1.0), ncol=2, fontsize=5.8, handletextpad=0.2, columnspacing=0.8)
letter(ax, "b", x=-0.55)

# (c) the strongest connection between and within the hemispheres (share of the squared weight), coloured by age
ax = fig.add_subplot(gs[2]); ax.scatter(d.top_inter, d.top_intra, **AGE)
lim = 0.5; ax.plot([0, lim], [0, lim], color=INK2, lw=0.5, ls=":")
ax.set(xlabel="between hemisph.", ylabel="within a hemisphere", xlim=(0, 0.62), ylim=(0, 0.32), xticks=[0, 0.3, 0.6], yticks=[0, 0.15, 0.3])
letter(ax, "c", x=-0.6)
for a in fig.axes:
    if a is not cax:
        sns.despine(ax=a)
save(fig, "fig6_interhemispheric")

## SI. Development, within cohorts

In [ ]:
fig = plt.figure(figsize=(W1, 4.3))
gs = fig.add_gridspec(2, 2, left=0.16, right=0.985, top=0.97, bottom=0.09, hspace=0.5, wspace=0.55, height_ratios=[1.15, 1])
W = N1["within"]; coh = [c for c in ORDER if f"{c}|MC" in W]
# (a) age trends within each cohort (95% CI)
ax = fig.add_subplot(gs[0, 0]); y = np.arange(len(coh))[::-1]
for off, col, key, lab in ((0.16, INK, "MC", "MC"), (-0.16, "#6a3d9a", "hom_F2", "$h$")):
    for yi, c in zip(y, coh):
        w = W[f"{c}|{key}"]; ax.plot([w["lo"], w["hi"]], [yi + off] * 2, color=col, lw=0.8); ax.plot(w["r"], yi + off, "o", color=col, ms=2.6)
    ax.plot([], [], "o-", color=col, ms=2.6, lw=0.8, label=lab)
ax.axvline(0, color=INK2, lw=0.5); ax.set_yticks(y); ax.set_yticklabels(coh, fontsize=6.2); ax.tick_params(axis="y", length=0)
for t, c in zip(ax.get_yticklabels(), coh): t.set_color(COH[c])
ax.set(xlabel="correlation with age\nwithin the cohort", xlim=(-0.75, 0.75)); ax.legend(loc="upper right"); letter(ax, "a", x=-0.45)
# (b) the turning point of memory within camCAN (single site)
ax = fig.add_subplot(gs[0, 1]); cc = d[d.cohort == "camCAN"]; tp = N1["camcan_turning_point"]["MC"]
ax.scatter(cc.age, cc.MC, s=1.2, color=COH["camCAN"], alpha=0.45, lw=0, rasterized=True)
bm = cc.groupby(pd.cut(cc.age, np.arange(15, 95, 5)), observed=True).agg(a=("age", "mean"), m=("MC", "mean"), s=("MC", "sem"))
ax.errorbar(bm.a, bm.m, yerr=1.96 * bm.s, color=INK, lw=0.9, marker="o", ms=2, capsize=1)
ax.axvspan(*tp["ci"], color="#d9d9d9", alpha=0.6, lw=0); ax.axvline(tp["age"], color=INK, lw=0.7, ls="--")
ax.text(tp["age"] + 1, ax.get_ylim()[1], f"{tp['age']:.0f} y", va="top", fontsize=6.3)
ax.set(xlabel="age (years)", ylabel="MC (camCAN)", xticks=[20, 40, 60, 80]); letter(ax, "b")
# (c) young adults of two cohorts at the same ages
for j, (col, lab) in enumerate((("MC", "memory capacity"), ("hom_F2", "homotopic share $h$"))):
    ax = fig.add_subplot(gs[1, j]); m = d[(d.age >= 22) & (d.age <= 37) & d.cohort.isin(["HCPya", "camCAN"])]
    sns.boxplot(data=m, x="cohort", y=col, order=["HCPya", "camCAN"], color="white", width=0.45, fliersize=0, linewidth=0.6, ax=ax,
                boxprops=dict(edgecolor=INK2), whiskerprops=dict(color=INK2), medianprops=dict(color=INK), capprops=dict(color=INK2))
    sns.stripplot(data=m, x="cohort", y=col, order=["HCPya", "camCAN"], hue="cohort", palette=COH, size=1.4, jitter=0.18, alpha=0.6, ax=ax, legend=False, rasterized=True)
    ax.set(xlabel="ages 22-37", ylabel=lab); ax.tick_params(axis="x", length=0); letter(ax, "c" if j == 0 else "d", x=-0.38)
for a in fig.axes:
    sns.despine(ax=a)
save(fig, "figS_development")